# 07 · Idempotence and Kafka transactions

**Level:** advanced · **Time:** 50–65 minutes · **Prerequisites:** notebooks 01–06.

You will distinguish producer retries from application duplicates, inspect aborted
transactions, and atomically publish output **and** commit input offsets.
Start Kafka with `docker compose up -d --wait` from the repository root.
Restart the kernel, then run cells from top to bottom. Each run creates fresh topics.

Keep these three boundaries separate: a producer's retry, a Kafka transaction,
and an external database transaction. They solve different failure cases.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


In [ ]:
import json
from confluent_kafka import KafkaException, TopicPartition

require_broker()
duplicate_topic = new_topic("07.duplicates", partitions=1)
visibility_topic = new_topic("07.visibility", partitions=1)
source_topic = new_topic("07.input", partitions=1)
output_topic = new_topic("07.output", partitions=1)


## 1. An idempotent producer does not deduplicate your business events

Our producer helper enables `enable.idempotence=True` and `acks=all`.
Send the same event twice deliberately. Both calls are legitimate new sends;
Kafka cannot infer that the second application call was a mistake.


In [ ]:
event = {"event_id": "payment-001", "amount_paise": 12500}
publish(duplicate_topic, [event, event], key_field="event_id")
duplicates = collect(duplicate_topic, count=2)
print([(m.offset(), decode(m)) for m in duplicates])
assert len({decode(m)["event_id"] for m in duplicates}) == 1
assert len(duplicates) == 2


**Observe:** two offsets, one business ID. Producer idempotence handles eligible
protocol retries; an event ID plus durable application deduplication handles repeated
application submissions. Do not use an in-memory set as a production deduplication store.


## 2. Publish one aborted and one committed transaction

A `transactional.id` identifies a logical writer. In a deployed service it must remain
stable across that writer's restart, while distinct concurrent writers need distinct IDs.
The random ID below is appropriate for this isolated notebook run. Reusing an ID from a
second producer can fence the first writer.


In [ ]:
tx = producer(**{"transactional.id": f"lab-07-{uuid4().hex}",
                 "transaction.timeout.ms": 30000})
tx.init_transactions(20)

def enqueue_checked(p, topic, values):
    errors = []
    def delivered(err, message):
        if err is not None:
            errors.append(str(err))
    for value in values:
        p.produce(topic, key=value["event_id"],
                  value=json.dumps(value).encode(), on_delivery=delivered)
        p.poll(0)
    remaining = p.flush(15)
    if remaining or errors:
        raise RuntimeError(f"Unconfirmed delivery: pending={remaining}, errors={errors}")


In [ ]:
tx.begin_transaction()
try:
    enqueue_checked(tx, visibility_topic,
                    [{"event_id": "aborted", "status": "do-not-use"}])
    # Flush first so this record reaches the log before we abort it.
    tx.abort_transaction(15)
except Exception:
    try:
        tx.abort_transaction(15)
    except KafkaException:
        pass
    raise
print("First transaction aborted.")


In [ ]:
tx.begin_transaction()
try:
    enqueue_checked(tx, visibility_topic,
                    [{"event_id": "committed-1", "status": "ready"},
                     {"event_id": "committed-2", "status": "ready"}])
    tx.commit_transaction(15)
except Exception:
    try:
        tx.abort_transaction(15)
    except KafkaException:
        pass
    raise
print("Second transaction committed.")


## 3. Compare the two isolation levels

`read_uncommitted` may return aborted records. `read_committed` hides them and
waits behind an open transaction's visibility boundary. Transaction control records
also consume offsets, so offsets need not be consecutive in the returned application records.


In [ ]:
raw = collect(visibility_topic, count=3,
              config={"isolation.level": "read_uncommitted"})
committed = collect(visibility_topic, count=2,
                    config={"isolation.level": "read_committed"})
print("Raw:", [(m.offset(), decode(m)["event_id"]) for m in raw])
print("Committed:", [(m.offset(), decode(m)["event_id"]) for m in committed])
assert {decode(m)["event_id"] for m in committed} == {"committed-1", "committed-2"}


## 4. Consume → transform → produce as one Kafka transaction

Publishing output and then separately committing source offsets leaves a crash gap.
`send_offsets_to_transaction` joins the source offsets to the output transaction.
We will abort once, restart the consumer, and then commit the replayed input.


In [ ]:
publish(source_topic, [
    {"event_id": "a", "amount_paise": 10000},
    {"event_id": "b", "amount_paise": 20000},
], key_field="event_id")
pipeline_group = f"lab-07-pipeline-{uuid4().hex}"

def transform_transaction(c, messages, *, abort):
    tx.begin_transaction()
    try:
        transformed = [{**decode(m), "tax_paise": decode(m)["amount_paise"] // 10}
                       for m in messages]
        enqueue_checked(tx, output_topic, transformed)
        # Offsets mean the NEXT record to consume, independently for each partition.
        next_offsets = {}
        for m in messages:
            k = (m.topic(), m.partition())
            next_offsets[k] = max(next_offsets.get(k, 0), m.offset() + 1)
        offsets = [TopicPartition(t, p, o) for (t, p), o in next_offsets.items()]
        tx.send_offsets_to_transaction(offsets, c.consumer_group_metadata(), 15)
        if abort:
            tx.abort_transaction(15)
        else:
            tx.commit_transaction(15)
    except Exception:
        try:
            tx.abort_transaction(15)
        except KafkaException:
            pass
        raise


In [ ]:
first = consumer(pipeline_group, **{"isolation.level": "read_committed"})
try:
    first.subscribe([source_topic])
    attempt_one = poll_records(first, count=2, timeout=20)
    transform_transaction(first, attempt_one, abort=True)
    print("Aborted input offsets:", [m.offset() for m in attempt_one])
finally:
    first.close()


The aborted transaction advanced neither the committed source offset nor the visible
output. The previous consumer's in-memory position did advance; closing it and using
the same group makes the restart behavior explicit.


In [ ]:
retry = consumer(pipeline_group, **{"isolation.level": "read_committed"})
try:
    retry.subscribe([source_topic])
    attempt_two = poll_records(retry, count=2, timeout=20)
    assert [m.offset() for m in attempt_one] == [m.offset() for m in attempt_two]
    transform_transaction(retry, attempt_two, abort=False)
    offsets_after = retry.committed([TopicPartition(source_topic, 0)], timeout=10)
    assert offsets_after[0].offset == 2
    print("Committed next input offset:", offsets_after[0].offset)
finally:
    retry.close()


In [ ]:
visible_output = collect(output_topic, count=2,
                         config={"isolation.level": "read_committed"})
visible_values = [decode(m) for m in visible_output]
print(visible_values)
assert {v["event_id"]: v["tax_paise"] for v in visible_values} == {"a": 1000, "b": 2000}
all_output = collect(output_topic, count=4,
                     config={"isolation.level": "read_uncommitted"})
assert len(all_output) == 4
print("Four physically sent output records; two transactionally visible results.")


## 5. State the guarantee precisely

This demonstrates atomic Kafka output plus Kafka consumer offsets for the input batch.
It does **not** atomically include a Django database update, HTTP call, email, or a
duplicate already present in the input topic. Use an outbox and durable idempotent
handling for database integration (notebooks 12–14).

In production inspect `KafkaException.args[0]`: a retriable error may permit retrying
the operation; `txn_requires_abort()` requires aborting and replaying the input;
fatal/fencing errors require stopping that writer. A commit timeout must not trigger a
blind new business send. This teaching code stops on errors instead of implementing a
deployment-specific recovery loop. Keep processing within `max.poll.interval.ms` and
the transaction timeout, and handle rebalances before committing stale ownership.


## Practice

1. Add a third input event and update the assertions.
2. Deliberately send event `a` twice to the input. Does a Kafka transaction remove it?
3. Explain why an HTTP call inside `transform_transaction` is not rolled back by abort.
4. Open a transaction, publish, and compare the two isolation levels before aborting.
   Use a short bounded poll for the committed reader; always abort in `finally`.

**Solution hints:** the source offset becomes 3; business duplicates remain legitimate
source records; external services are outside Kafka's transaction coordinator; a
committed reader can wait for an earlier transaction to resolve.


## Presenter checkpoint and references

Ask a learner to explain all three observations: application duplicates survived,
aborted output was hidden, and aborted source offsets replayed.

Reference APIs: [Confluent Python transactions](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html#transactional-api),
[official consume/produce example](https://github.com/confluentinc/confluent-kafka-python/blob/master/examples/eos_transactions.py).
